# **Machine Failure Prediction for TATA Steel — Exploratory Data Analysis (EDA Capstone)**

##### **Project Type**    - EDA/Regression/Classification/Unsupervised
##### **Contribution**    - Individual
##### **Name -**  Tushar


# **Project Summary -**

This EDA project explores TATA Steel's synthetic manufacturing sensor dataset to understand the conditions under which machinery fails. The training data has 136,429 rows and 14 columns, with zero missing values and zero duplicate rows. The target, `Machine failure`, is severely imbalanced: only 2,148 records (1.57%) are actual failures. Machines fall into three quality tiers (`Type`: L, M, H), and five specific failure-mode flags (`TWF`, `HDF`, `PWF`, `OSF`, `RNF`) are recorded alongside five continuous sensor readings (Air temperature, Process temperature, Rotational speed, Torque, Tool wear). After dropping the non-informative `id` and `Product ID` columns (the latter's first letter simply duplicates `Type`), fifteen visualizations were built to understand class imbalance, sensor distributions, and how each sensor relates to failure. The clearest findings: Torque and Tool Wear separate failed machines from healthy ones more than any other sensor; Air and Process temperature are almost perfectly correlated with each other; Heat Dissipation Failure (HDF) is the single most common failure sub-type; and Low-tier (L) machines show a higher failure rate than Medium or High-tier machines. These findings translate directly into a business recommendation: prioritize monitoring Torque and Tool Wear in real time, invest in cooling/heat dissipation improvements, and consider tighter quality control for Low-tier machines — and treat any future predictive model's accuracy with caution given how rare failures are.

# **GitHub Link -**

Add your GitHub repository link here once you have committed this notebook, e.g. `https://github.com/<your-username>/tata-steel-machine-failure-eda`

# **Problem Statement**


**Explore TATA Steel's manufacturing sensor data to understand what operating conditions are associated with machine failure, and identify which sensors and machine attributes most strongly distinguish a failed machine from a healthy one — without yet building a predictive model.**

#### **Define Your Business Objective?**

**To help TATA Steel move toward predictive, rather than purely reactive, maintenance by identifying which measurable operating conditions (temperature, speed, torque, tool wear, machine type) are most associated with machine failure, so maintenance and engineering teams know where to focus monitoring and process improvements.**

# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required.
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits.
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 20 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule.

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

### Dataset Loading

In [ ]:
# Load Dataset
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
print('Train shape:', train.shape)
print('Test shape :', test.shape)

### Dataset First View

In [ ]:
# Dataset First Look
train.head()

### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
print('Train -> Rows:', train.shape[0], '| Columns:', train.shape[1])
print('Test  -> Rows:', test.shape[0], '| Columns:', test.shape[1])

### Dataset Information

In [ ]:
# Dataset Info
train.info()

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
print('Duplicate rows in train:', train.duplicated().sum())
print('Duplicate rows in test :', test.duplicated().sum())

#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count
print(train.isnull().sum())
print('\nTotal missing values in train:', train.isnull().sum().sum())

In [ ]:
# Visualizing the missing values
plt.figure(figsize=(10,4))
sns.heatmap(train.isnull(), cbar=False, cmap='viridis')
plt.title('Missing Value Map - Train (fully clean, no gaps expected)')
plt.show()

### What did you know about your dataset?

The training set has **136,429 rows and 14 columns** with **zero missing values and zero duplicates**. The target `Machine failure` is heavily imbalanced: **98.43% healthy vs 1.57% failed** (2,148 of 136,429 records). Machines split into three types: L (95,354), M (32,152), H (8,923). This is a clean but rare-event dataset — the core analytical challenge is understanding a pattern that occurs in less than 2% of records.

## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
train.columns.tolist()

In [ ]:
# Dataset Describe
train.describe()

### Variables Description

- `id` — row identifier, no analytical value.
- `Product ID` — per-unit identifier; first letter always equals `Type`, so it's redundant.
- `Type` — machine quality tier: L (Low), M (Medium), H (High).
- `Air temperature [K]`, `Process temperature [K]` — temperature sensors in Kelvin; strongly correlated with each other.
- `Rotational speed [rpm]` — spindle speed.
- `Torque [Nm]` — mechanical torque applied.
- `Tool wear [min]` — cumulative minutes the cutting tool has been used.
- `TWF, HDF, PWF, OSF, RNF` — binary flags for five specific failure mechanisms.
- `Machine failure` — the target (1 = failed).

### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.
for col in train.columns:
    print(f'{col:28s} -> {train[col].nunique()} unique values')

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Write your code to make your dataset analysis ready.
train_eda = train.drop(columns=['id', 'Product ID'])

# Confirm Product ID's first letter always matches Type (proves it's safe to drop)
print('Product ID prefix always equals Type:', train['Product ID'].str[0].eq(train['Type']).all())
train_eda.head()

### What all manipulations have you done and insights you found?

Dropped `id` (pure row index) and `Product ID` (its first letter is always identical to `Type`, and its numeric suffix is just a serial number with no analytical value). No missing-value imputation or de-duplication was required since none existed. This leaves 12 usable columns for exploration.

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1

In [ ]:
# Chart - 1 visualization code
plt.figure(figsize=(5,4))
sns.countplot(x='Machine failure', data=train)
plt.title('Target Class Balance')
plt.show()
train['Machine failure'].value_counts(normalize=True)*100

##### 1. Why did you pick the specific chart?

A count plot is the simplest way to show class balance for a binary outcome.

##### 2. What is/are the insight(s) found from the chart?

Only 1.57% of records are failures (2,148 of 136,429) — a severe class imbalance.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes — any future model or manual monitoring rule must account for this; accuracy alone would be a misleading success metric, and even small numbers of missed failures matter operationally.

#### Chart - 2

In [ ]:
# Chart - 2 visualization code
plt.figure(figsize=(5,4))
sns.countplot(x='Type', data=train, order=['L','M','H'])
plt.title('Machine Type Distribution')
plt.show()

##### 1. Why did you pick the specific chart?

A count plot shows the fleet composition across the three quality tiers.

##### 2. What is/are the insight(s) found from the chart?

Low-tier (L) machines dominate the fleet (95,354), followed by Medium (32,152) and High (8,923).

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes — since most of the fleet is Low-tier, understanding failure behavior specifically for that tier has outsized business impact.

#### Chart - 3

In [ ]:
# Chart - 3 visualization code
plt.figure(figsize=(6,4))
sns.histplot(train['Air temperature [K]'], kde=True, bins=40)
plt.title('Air Temperature Distribution')
plt.show()

##### 1. Why did you pick the specific chart?

A histogram with KDE shows the shape and spread of a continuous sensor reading.

##### 2. What is/are the insight(s) found from the chart?

Air temperature is roughly normally distributed between ~295K and ~304K, centered near 300K.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Limited on its own — a tight, predictable range means large deviations could matter, but Chart 10 shows this sensor barely separates failed from healthy machines.

#### Chart - 4

In [ ]:
# Chart - 4 visualization code
plt.figure(figsize=(6,4))
sns.histplot(train['Process temperature [K]'], kde=True, bins=40, color='orange')
plt.title('Process Temperature Distribution')
plt.show()

##### 1. Why did you pick the specific chart?

Same rationale as Chart 3, applied to Process temperature.

##### 2. What is/are the insight(s) found from the chart?

Also roughly normal (~305K-314K) and tracks Air temperature closely (confirmed in Chart 14).

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Indirectly — the strong relationship with Air temperature means the two carry mostly duplicate information for spotting failures.

#### Chart - 5

In [ ]:
# Chart - 5 visualization code
plt.figure(figsize=(6,4))
sns.histplot(train['Rotational speed [rpm]'], kde=True, bins=40, color='green')
plt.title('Rotational Speed Distribution')
plt.show()

##### 1. Why did you pick the specific chart?

A histogram reveals skew a plain describe() table can hide.

##### 2. What is/are the insight(s) found from the chart?

Rotational speed is right-skewed with a long tail toward higher RPM values.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes — unusually high RPM excursions are worth watching operationally as a possible early warning sign.

#### Chart - 6

In [ ]:
# Chart - 6 visualization code
plt.figure(figsize=(6,4))
sns.histplot(train['Torque [Nm]'], kde=True, bins=40, color='purple')
plt.title('Torque Distribution')
plt.show()

##### 1. Why did you pick the specific chart?

A histogram checks Torque's shape before comparing it against the target.

##### 2. What is/are the insight(s) found from the chart?

Torque is roughly normally distributed around ~40 Nm.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes — establishing the normal operating band helps define a sensible alert threshold later.

#### Chart - 7

In [ ]:
# Chart - 7 visualization code
plt.figure(figsize=(5,4))
sns.boxplot(x='Machine failure', y='Torque [Nm]', data=train)
plt.title('Torque vs Machine Failure')
plt.show()

##### 1. Why did you pick the specific chart?

A boxplot compares a numeric feature's distribution across the two target classes directly.

##### 2. What is/are the insight(s) found from the chart?

Failed machines show a visibly higher and wider-spread Torque distribution — the clearest visual separation of any single sensor.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes — Torque is the strongest single candidate for a real-time maintenance alert.

#### Chart - 8

In [ ]:
# Chart - 8 visualization code
plt.figure(figsize=(5,4))
sns.boxplot(x='Machine failure', y='Tool wear [min]', data=train)
plt.title('Tool Wear vs Machine Failure')
plt.show()

##### 1. Why did you pick the specific chart?

Boxplot comparison applied to Tool Wear, a classic mechanical-failure driver.

##### 2. What is/are the insight(s) found from the chart?

Failed machines skew toward higher tool wear values.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes — supports scheduling tool replacement after a defined wear threshold rather than running tools to failure.

#### Chart - 9

In [ ]:
# Chart - 9 visualization code
plt.figure(figsize=(5,4))
sns.boxplot(x='Machine failure', y='Rotational speed [rpm]', data=train)
plt.title('Rotational Speed vs Machine Failure')
plt.show()

##### 1. Why did you pick the specific chart?

Completes the boxplot check across remaining sensors.

##### 2. What is/are the insight(s) found from the chart?

Separation is weaker than Torque/Tool Wear, though failed machines show a slightly wider RPM spread including some low-RPM outliers.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Moderate — RPM alone isn't a strong standalone signal but is still worth tracking alongside Torque.

#### Chart - 10

In [ ]:
# Chart - 10 visualization code
plt.figure(figsize=(5,4))
sns.boxplot(x='Machine failure', y='Air temperature [K]', data=train)
plt.title('Air Temperature vs Machine Failure')
plt.show()

##### 1. Why did you pick the specific chart?

Same boxplot approach, completing the sweep across all continuous sensors.

##### 2. What is/are the insight(s) found from the chart?

Air temperature shows only a small shift between failed and healthy machines — the weakest separator among the sensors.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Limited — not a strong standalone alert trigger on its own.

#### Chart - 11

In [ ]:
# Chart - 11 visualization code
plt.figure(figsize=(5,4))
sns.barplot(x='Type', y='Machine failure', data=train, order=['L','M','H'], estimator=np.mean)
plt.title('Failure Rate by Machine Type')
plt.ylabel('Failure Rate')
plt.show()
train.groupby('Type')['Machine failure'].mean()*100

##### 1. Why did you pick the specific chart?

A bar plot of the mean of a 0/1 target shows failure *rate* per category, unlike a raw count plot.

##### 2. What is/are the insight(s) found from the chart?

Failure rate differs by machine tier, with Low-tier (L) machines generally showing a higher failure rate than Medium/High-tier machines.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes, directly — supports prioritizing maintenance budget and monitoring toward the Low tier.

#### Chart - 12

In [ ]:
# Chart - 12 visualization code
plt.figure(figsize=(6,4))
train[['TWF','HDF','PWF','OSF','RNF']].sum().sort_values().plot(kind='barh', color='teal')
plt.title('Frequency of Each Failure Sub-Type')
plt.xlabel('Count')
plt.show()

##### 1. Why did you pick the specific chart?

A horizontal bar chart ranks the five failure sub-types, easier to read than a table.

##### 2. What is/are the insight(s) found from the chart?

Heat Dissipation Failure (HDF, 704 cases) is the most common, followed by Overstrain (OSF, 540), Power Failure (PWF, 327), Random Failure (RNF, 308), and Tool Wear Failure (TWF, 212).

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes — a concrete maintenance priority: cooling/heat-dissipation systems are the single biggest contributor to failures.

#### Chart - 13

In [ ]:
# Chart - 13 visualization code
plt.figure(figsize=(6,5))
sample = train.sample(5000, random_state=42)
sns.scatterplot(x='Air temperature [K]', y='Process temperature [K]', hue='Machine failure',
                data=sample, alpha=0.5, palette={0:'steelblue', 1:'red'})
plt.title('Air vs Process Temperature (colored by failure)')
plt.show()

##### 1. Why did you pick the specific chart?

A scatterplot visually confirms a suspected near-linear relationship between the two temperature sensors.

##### 2. What is/are the insight(s) found from the chart?

The two temperatures move almost perfectly together; failures (red) don't cluster in a distinct temperature region.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Indirectly — confirms temperature alone is a weak, redundant predictor of failure.

#### Chart - 14 - Correlation Heatmap

In [ ]:
# Correlation Heatmap visualization code
plt.figure(figsize=(9,7))
corr_cols = ['Air temperature [K]','Process temperature [K]','Rotational speed [rpm]',
             'Torque [Nm]','Tool wear [min]','TWF','HDF','PWF','OSF','RNF','Machine failure']
sns.heatmap(train[corr_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

##### 1. Why did you pick the specific chart?

A correlation heatmap checks pairwise linear relationships across every numeric feature at once.

##### 2. What is/are the insight(s) found from the chart?

Air and Process temperature are very highly correlated (~0.9+); `HDF` correlates most strongly with `Machine failure` among the failure flags; Rotational speed and Torque are moderately negatively correlated.

#### Chart - 15 - Pair Plot

In [ ]:
# Pair Plot visualization code
sample = train.sample(2000, random_state=42)
sns.pairplot(sample, hue='Machine failure',
             vars=['Torque [Nm]','Tool wear [min]','Rotational speed [rpm]'],
             palette={0:'steelblue', 1:'red'}, plot_kws={'alpha':0.5})
plt.show()

##### 1. Why did you pick the specific chart?

A pair plot shows pairwise relationships and class separation across the three most promising features at once.

##### 2. What is/are the insight(s) found from the chart?

Failures (red) cluster toward jointly higher Torque and Tool wear more than either feature shows individually, hinting that combining sensors would separate classes better than any single one.

## **5. Solution to Business Objective**

#### What do you suggest the client to achieve Business Objective ?
Explain Briefly.

TATA Steel should: **(1)** prioritize real-time monitoring of **Torque** and **Tool Wear**, the two sensors that most clearly separate failing machines from healthy ones; **(2)** investigate and invest in **cooling/heat-dissipation systems**, since Heat Dissipation Failure (HDF) is the single most frequent failure sub-type; **(3)** apply **tighter quality control or more frequent servicing to Low-tier (L) machines**, which show a higher failure rate than Medium/High-tier machines; and **(4)** when building any future predictive system on this data, evaluate it with Precision/Recall/F1 rather than accuracy, since the 1.57% failure rate makes accuracy a misleading metric — a model that always predicts "no failure" would look 98.4% accurate while catching zero real failures. No insights here point toward negative growth; all findings support concrete, low-risk operational improvements.

# **Conclusion**

This EDA established that TATA Steel's machine failure problem is a rare-event problem (1.57% positive rate) with no data quality issues (no missing values, no duplicates). Torque and Tool Wear are the strongest individual indicators of failure, Air and Process temperature are redundant with each other, and Heat Dissipation Failure is the dominant failure mode. Low-tier machines fail more often than Medium/High-tier machines. These findings define exactly which sensors and machine attributes should feed a follow-up predictive model (covered in the companion ML notebook) and which maintenance actions TATA Steel can take immediately.

### ***Hurrah! You have successfully completed your EDA Capstone Project !!!***